In [7]:
import pandas as pd
from pathlib import Path
import json

DATA_PATH = Path("./data")
kacs_transform = {
    0:"Algebra",
    1:"Geometry",
    2:"Algebra",
    3:"Combinatorics",
    4:"Number Theory",
    5:"Combinatorics",
    6:"Algebra"
}

def topics2topic(x) :
    x = x[2:-2].split(" > ")[0].replace("\n",'')
    if "Math Word Problems'" in x :
        x = x.split("' '")[1]
    return x

mathnet_transform = {
    'Discrete Mathematics' : "Combinatorics",
    'Precalculus' : "Algebra",
    'Statistics' : "Combinatorics",
    'Calculus' : "Algebra",
    'Geometry' : "Geometry",
    'Algebra' : "Algebra",
    'Number Theory' : "Number Theory"
}

In [2]:
file_kacs = DATA_PATH / "train_kacs.csv"
df_kacs = pd.read_csv(file_kacs)
df_kacs = df_kacs.drop(df_kacs[df_kacs["label"]==7].index)
df_kacs["topic"] = df_kacs["label"].map(kacs_transform)
df_kacs = df_kacs.drop(["label"],axis = 1)
df_kacs.columns = ["problem","topic"]


In [3]:
file_mathnet = DATA_PATH / "train_mathnet.csv"
df_mathnet = pd.read_csv(file_mathnet)
df_mathnet["topic"] = df_mathnet["topics_flat"].apply(topics2topic)
df_mathnet = df_mathnet.drop(df_mathnet[df_mathnet["topic"] == "Math Word Problems"].index)
df_mathnet["topic"] = df_mathnet["topic"].replace(mathnet_transform)
df_mathnet = df_mathnet.drop(["topics_flat"],axis = 1)
df_mathnet.columns = ["problem","topic"]


In [4]:
train_all = pd.concat([df_kacs,df_mathnet]).reset_index(drop=True)

In [5]:
solution_raw =  DATA_PATH / "solution_kacs.csv"
solution_raw = pd.read_csv(solution_raw)
solution = solution_raw.drop(solution_raw[solution_raw["label"]==7].index)
solution = solution.drop(["Usage"],axis = 1)
solution["label"] = solution["label"].map(kacs_transform)
solution

,id,label
0,0,Algebra
1,1,Number Theory
2,2,Algebra
3,3,Algebra
4,4,Number Theory
...,...,...
3039,3039,Number Theory
3040,3040,Geometry
3041,3041,Geometry
3042,3042,Number Theory


In [6]:
file_test =  DATA_PATH / "test_kacs.csv"
test = pd.read_csv(file_test)
test.columns = ["id","problem"]
test = test.drop(solution_raw[solution_raw["label"]==7].index)
test


,id,problem
0,0,b'Solve 0 = -i - 91*i - 1598*i - 64220 for i.\n'
1,1,Galperin G.A.\n\nA natural number $N$ is 999.....
2,2,Example 7 Calculate $\frac{1}{2 \sqrt{1}+\sqrt...
3,3,"If $A$, $B$, and $C$ represent three distinct ..."
4,4,2. Calculate $1+12+123+1234+12345+123456+12345...
...,...,...
3039,3039,"Find the greatest possible value of $pq + r$, ..."
3040,3040,"4. Given that $a, b, c$ are the lengths of the..."
3041,3041,"3.18. Find the eccentricity, coordinates of th..."
3042,3042,Find the least positive integer $k$ for which ...


In [7]:
train_all.to_csv(DATA_PATH / "train.csv",index = False)
test.to_csv(DATA_PATH / "test.csv",index = False)
solution.to_csv(DATA_PATH / "solution.csv",index = False)

In [10]:
OUTPUT_PATH = Path("output/")
contestpath = [item for item in OUTPUT_PATH.iterdir()]
full_problem = []

for contest in contestpath :
    file = contest / 'full_tex.json'
    with open(file,'r',encoding='utf-8') as f :
        data = json.load(f)
        full_problem+=data
df_full = pd.DataFrame(full_problem)

df_full = pd.DataFrame(full_problem)
df_full = df_full.loc[:,['id','contest','version','year','problem_number','topic','problem_statement_tex','answer','difficulty','solutions_statement_tex']]
df_full.to_excel('AMC_full.xlsx',index = False)

In [14]:
df_full["topic"].value_counts()

topic
Algebra          2918
Geometry         2133
Combinatorics    1282
Number Theory     624
Name: count, dtype: int64